In [ ]:
from joblib import Parallel, delayed
import pandas as pd
import gc
from pathlib import Path
import sys

# =========================
# PROJECT SETUP
# =========================
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "utils").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "utils").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data import load_clean_citation_dataframe_from_files, build_training_dataframe

# =========================
# PATHS
# =========================
OUTPUT_DIR = PROJECT_ROOT / "data" / "exploded_splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_SPLIT_DIR = PROJECT_ROOT / 'data' / 'exploded_splits'

SPLIT_FILES = {
    'train': DATA_SPLIT_DIR / 'train_pairs.parquet',
    'validation': DATA_SPLIT_DIR / 'validation_pairs.parquet',
    'test': DATA_SPLIT_DIR / 'test_pairs.parquet',
}

In [ ]:
# =========================
# PROCESS FUNCTION
# =========================
def process_and_save_split(split_name: str, seed: int) -> dict:
    raw_df = load_clean_citation_dataframe_from_files(
        [SPLIT_FILES[split_name]],
        remove_empty_rows=True
    )

    n_raw = len(raw_df)

    if raw_df.empty:
        return {
            "split": split_name,
            "status": "empty",
            "n_raw": 0,
            "n_final": 0
        }

    df = build_training_dataframe(
        raw_df,
        seed=seed,
        filter_years=True
    ).assign(split=split_name)

    df.dropna(subset=['ref_id'], inplace=True)
    df.drop(columns=['year', 'n_citation_ref'], inplace=True, errors="ignore")
    str_cols = df.select_dtypes(include=["object", "string"]).columns
    raw_df[str_cols] = df[str_cols].fillna("")

    n_final = len(df)

    output_path = OUTPUT_DIR / f"{split_name}_pairs.parquet"
    df.to_parquet(output_path, index=False)

    # =========================
    # CLEAN MEMORY
    # =========================
    del raw_df
    del df
    gc.collect()

    return {
        "split": split_name,
        "status": "ok",
        "n_raw": n_raw,
        "n_final": n_final,
        "ratio": round(n_final / n_raw, 3) if n_raw > 0 else 0,
        "output_path": str(output_path),
    }

# =========================
# SEQUENTIAL EXECUTION (ONE SPLIT AT A TIME)
# =========================
split_jobs = [('train', 42),('validation', 43), ('test', 44)]
results = []

for split_name, seed in split_jobs:
    print(f"\n{'='*50}")
    print(f"Processing: {split_name.upper()}")
    print(f"{'='*50}")
    
    result = process_and_save_split(split_name, seed)
    results.append(result)
    
    print(f"✓ {split_name}: {result['n_raw']} → {result['n_final']} rows")

In [ ]:
# =========================
# SUMMARY TABLE
# =========================
summary_df = pd.DataFrame(results)

print("\n=== DATASET SUMMARY ===")
print(summary_df[["split", "n_raw", "n_final", "ratio"]])

# =========================
# DETAILED PREVIEW
# =========================
for res in results:
    print(f"\n=== {res['split'].upper()} ===")
    print(f"Rows: {res['n_raw']} → {res['n_final']} (ratio={res.get('ratio', 'N/A')})")

print("\n=== DONE ===")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from utils import balance_classes

target_counts = {}

for split in SPLIT_FILES.keys():
    df = pd.read_parquet(OUTPUT_DIR / f"{split}_pairs.parquet")
    target_counts[split] = df['is_reference_valid'].value_counts().to_dict()
    del df
    
print(target_counts)

fig, axes = plt.subplots(1, 3, figsize=(12, 5), sharey=True)

for i, (split, counts) in enumerate(target_counts.items()):
    axes[i].bar(
        ['Valid', 'Invalid'],
        [counts.get(1, 0), counts.get(0, 0)],
        color=['green', 'red']
    )
    axes[i].set_title(split)

plt.tight_layout()
plt.show()